## Overview
### The What
Consider the scenario below:
- Application on `my-app.com` makes a JavaScript request to `api.com/data`
- The server on `api.com` returns a response back to browser
- The browser checks the header `Access-Control-Allow-Origin` to see if it contains `my-app.com`
- If yes, the browser accepts the response; if no, the browser rejects the response

![CORS Overview](./images/cors.png)

What about potentially mutating requests like `PUT`, `DELETE`, etc.? By the time the browser rejects the response, the server would have already processed the request. To prevent this, before sending such a request, the browser first sends a preflight request (an `OPTIONS` request). It then checks the `Access-Control-Allow-Origin` header (among others) in the preflight response, and depending on the contents of that header, either proceeds with the actual request or blocks it entirely.

![Preflight](./images/preflight.png)

What are the scenarios where CORS is enforced?
- `fetch()` / `XMLHttpRequest` to another origin
- Web fonts (`@font-face`) from another origin
- `<script type="module">` from another origin
- Resources fetched by HTML elements with the `crossorigin` attribute
Plain `<img>`, `<script>`, `<link rel=stylesheet>`, `<video>` don't do CORS check.

What are the scenarios when preflight request is made? All of the below need to be true for no preflight:
- `GET`, `HEAD` or `POST` request
- Request only has basic headers which are : `Accept`, `Accept-Language`, `Content-Language`, `Content-Type` (only with the 3 values: `application/x-www-form-urlencoded`, `multipart/form-data`, `text/plain`) and `Range`. This list is in addition to browser added headers like `Cookie`, `User-Agent`, `Host`, `Connnection` and `Origin`.

The examples below illustrate the point:

In [ ]:
// No preflight (all headers are safe)
fetch("https://api.com/data", {
  headers: { "Accept": "application/json" }
});

// Preflight (Authorization is not on the list)
fetch("https://api.com/data", {
  headers: { "Authorization": "Bearer abc" }
});

// Preflight (custom header)
fetch("https://api.com/data", {
  headers: { "X-Api-Key": "123" }
});

// Preflight (Content-Type not on the allowed values)
fetch("https://api.com/data", {
  method: "POST",
  headers: { "Content-Type": "application/json" }
});

**A note on Origin:** it is `scheme` + `host` + `port`. Example:
```
https://www.wikipedia.org:443
  |             |          |
scheme         host       port
```

The browser does exact comparison between the current pages origin (what you see in the address bar) against what we get in `Access-Control-Allow-Origin` header. It is common for the server to have both `https://my-app.com` and `https://www.my-app.com` in the header.

This exact comparison is also why UI on `localhost:5000` enforces CORS on backend requests to `localhost:8080`.

## The Why
Imagine the scenario where you are on an attacker's website `evil.com`:
- web content on `evil.com` makes a request `fetch('bank.com/account')`
- the browser automatically attaches cookies for `bank.com` in the outgoing request
- `bank.com` returns your account information which is then read by the attacker's website

CORS prevents this scenario. Origin `evil.com` is not returned in the `Access-Control-Allow-Origin` header; thus the response is rejected by the browser.  
**Note:** appropriate `SameSite` cookie policy can also prevent this exploit by not attaching the cookie.

## Server Changes
The server needs to do two things:
- add the `Access-Control-Allow-Origin` header to all requests it receives
- respond similarly to `OPTIONS` request

In [ ]:
const allowed = ["https://my-app.com", "https://www.my-app.com"];

app.use((req, res, next) => {
  const origin = req.headers.origin;

  // Access-Control-Allow-Origin can have only one value (and not an array).
  // Which is why have to do this check (and not add the allowed list to all requests)  
  if (allowed.includes(origin)) {
    res.setHeader("Access-Control-Allow-Origin", origin);
    res.setHeader("Vary", "Origin");
    res.setHeader("Access-Control-Allow-Credentials", "true");
  }

  if (req.method === "OPTIONS") {            // preflight
    res.setHeader("Access-Control-Allow-Methods", "GET,POST,PUT,DELETE");
    res.setHeader("Access-Control-Allow-Headers", "Content-Type,Authorization");
    res.setHeader("Access-Control-Max-Age", "600");
    return res.sendStatus(204);
  }

  next();
});

`Vary` header tells the caches - “This response depends on the value of these request headers. Don’t reuse it for a request that has a different value.” By default, a cache (browser, CDN, proxy) keys a response by the URL only. Without it:

```
1. https://my-app.com requests /data
   Server: Access-Control-Allow-Origin: https://my-app.com
   CDN caches this response for the URL /data

2. https://www.my-app.com requests /data
   CDN sees the same URL -> serves the cached response
   Header says: https://my-app.com
   Browser compares with https://www.my-app.com -> mismatch -> CORS error

```

As we can see in the code above, there are other `Access-Control-Allow-*` headers as well: